# 56 · Deep conditioning **trên kiến trúc chính** — `39` đo trong điều kiện đọc hụt EHR

## Vì sao chạy lại thứ `39` đã đo

`39` cho `G_film = +0.0020` (vùng xám, 1 seed). Nhưng nó đo trên `mldec + adapter` — **không có
`Tier1Img`, không có phim trước**. Và ta vừa có bằng chứng rằng chính điều kiện đó làm EHR đọc hụt:

```
52b  mldec + Tier1, KHONG Tier1Img   ->  noi dung(col) = -0.0015
53   + Tier1Img, CO phim truoc       ->  noi dung(col) = +0.0060
```

**Cùng bộ mã hoá, cùng dữ liệu EHR, hai dấu ngược nhau — khác biệt duy nhất là nhánh phim trước.**
Nên `+0.0020` của `39` rất có thể là **đọc hụt**, không phải giá trị thật.

*(Suy luận **qua notebook** — `52b` và `53` khác nhau nhiều thứ ngoài `Tier1Img`. Nhưng hướng nhất
quán và đủ mạnh để đáng một lần chạy.)*

## Năm arm — và câu hỏi `39` KHÔNG trả lời được

```
mldec                   goc toa do                             | fwd CHUNG
t1_prior_both           KIEN TRUC CHINH (53 do 0.2560, 2 seed)  | fwd CHUNG
t1_prior_both_ehrz      doi chung HANG SO cho nhanh EHR         | fwd CHUNG
t1_prior_both_film      + FiLM adapter sau block 8, 9           | fwd RIENG tu block 8
t1_prior_both_film_u    adapter nhan HANG SO (Tier1 van thuc)   | fwd RIENG

G_ehr     = both      - both_ehrz     lap lai +0.0060 tren CUNG lan chay
G_deep    = both_film - both_film_u   deep conditioning co NOI DUNG khong
CONG THEM = both_film - both          <- CAU HOI CHINH: deep them duoc gi TREN kien truc chinh
```

`39` không có `t1_prior_both` để so nên không trả lời được dòng cuối.

## Một quyết định thiết kế dễ sai

Adapter áp cho **CẢ hai ảnh** (hiện tại **và** phim trước). `Tier1Img(diff=True)` tính `patch − a`;
nếu chỉ `patch` bị điều biến còn `a` thì không, phép hiệu chứa **chính phần điều biến** trộn với
"cái gì đã thay đổi" ⇒ nhánh phim trước bị nhiễm. Tốn gấp đôi phần backbone tính lại, nhưng không
có lựa chọn nào sạch hơn.

## Đọc theo TIER, không chỉ macro

`39` cho `tail −0.0050`. `53` cho `t1_prior_both` tail `0.1077` — cao nhất dự án. Nếu deep lại kéo
tail xuống thì **đóng dứt điểm**, bất kể macro.

## C0 · Config

In [ ]:
import os, json, math, time, tarfile, io as _io, re, datetime
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024'); EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RUN_TAG = 'r1'
RES56 = os.path.join(OUT_DIR, 'stage4_nb56', RUN_TAG); CKPT = os.path.join(RES56, 'ckpt')
P_OOF = os.path.join(OUT_DIR, 'stage4_nb52a', 'xgb_oof.npy')
P_TOUCH = os.path.join(OUT_DIR, '_test_touches.jsonl')
def ensure_res():
    '''Drive FUSE danh roi thu muc RONG — goi TRUOC moi cum ghi (43/38c/38d/41a da dinh).'''
    os.makedirs(RES56, exist_ok=True); os.makedirs(CKPT, exist_ok=True); return RES56
ensure_res()
json.dump({'run_tag': RUN_TAG, 'notebook': '56'}, open(os.path.join(RES56, '_run_info.json'), 'w'))
open(os.path.join(CKPT, '_keep'), 'w').write('keep')
LOCAL = '/content/cxrlt_518'

ADAPT_AFTER = (8, 9)     # `11`: I_location ~ 0 giua (5,7) va (10,11). TRANH 11-12 (NOTE_07:
                         # adapter o block cuoi la "late fusion doi lot")
FILM_HID = 256           # giu dang bottleneck cua `11` (234,368 tham so) de so duoc
BB_RUN = 'a2_E3'; SEEDS = (42, 43)
SEED, SIZE, EPOCHS = 42, 518, 10          # E=10: `53` do lat chon binh nguyen ep8-10 (0.2451-0.2452)
BATCH, ACCUM = 32, 8                      # 2 anh/mau + graph block 8-11 x2 arm. EFF 256.
LR, WD, WARMUP, CLIP = 1e-3, 1e-2, 0.05, 5.0   # GIU y het 53 de so truc tiep voi moc 0.2560
W_FRONTAL, W_LATERAL = 0.7, 0.3
NW = min(8, (os.cpu_count() or 2))
D_MLD, D_ETOK, EHR_CLIP = 512, 256, 10.0
DO_TEST, N_TEST_EP = True, 3

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'thiet bi: {torch.cuda.get_device_name(0)} | '
      f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP {AMP}')

BB_SRC = os.path.join(OUT_DIR, 'stage4_nb38a', BB_RUN, 'backbone_ft.pt')
P_MICRO = os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet')
P_VOCAB = os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json')
for p, hd in ((BB_SRC, '38a'), (P_MICRO, '29c'), (P_VOCAB, '29c'), (P_OOF, '52a')):
    assert os.path.exists(p), f'khong thay {p} -> chay {hd} truoc'

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')
P_VIEW = os.path.join(OUT_DIR, 'cxrlt_view_resolved.parquet')
V_ = ((pd.read_parquet(P_VIEW)[['dicom_id','view_group','use']]
         .set_index('dicom_id').loc[ORDER.dicom_id].reset_index()) if os.path.exists(P_VIEW)
      else ORDER[['dicom_id','view_group']].assign(use=lambda d: d.view_group != 'unknown'))
SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'; TE = ~(TR | VA)
_rs = np.random.default_rng(SEED); _su = np.unique(SUB[TR])
_sel = _rs.choice(_su, size=int(round(0.15*len(_su))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL      # CUNG cach cat nhu 35/37/38*/44*/45/52*/53
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
I_TF, I_SL, I_VA, I_TE = (np.where(m)[0] for m in (TF_, SL, VA, TE))
print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,} | '
      f'test {len(I_TE):,}')
print(f'adapter SAU block {ADAPT_AFTER} | E={EPOCHS} | BATCH {BATCH}x{ACCUM} | seeds {SEEDS}')

## C1 · EHR + túi vi khuẩn + XGB + phim trước — verbatim `53`

In [ ]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()
COLS = [c for c in SPEC['Q_FULL'] if c in E_DF.columns]
_m = np.nan_to_num((E_DF.set_index('study_id')[COLS].reindex(_sid)
                    .apply(pd.to_numeric, errors='coerce').to_numpy('float32')), nan=0.0)
# Cot NHI PHAN thi KHONG z-score: p = 1/219,785 -> gia tri chuan hoa 468 -> softmax bao hoa -> NaN
_b = np.array([np.isin(np.unique(_m[:, j]), (0.0, 1.0)).all() for j in range(_m.shape[1])])
_mu = _m[TF_].mean(0, keepdims=True); _sd = _m[TF_].std(0, keepdims=True) + 1e-6
_mu[:, _b] = 0.0; _sd[:, _b] = 1.0
E_ALL = np.clip((_m - _mu) / _sd, -EHR_CLIP, EHR_CLIP).astype('float32')
_keep = [i for i, c in enumerate(COLS) if not re.match(r'^micro_org_', c)]
assert len(COLS) - len(_keep) == 30, f'bo {len(COLS)-len(_keep)} cot micro_org_*, phai la 30'
COLS = [COLS[i] for i in _keep]; EHR_COL = E_ALL[:, _keep]

_MIC = pd.read_parquet(P_MICRO).set_index('study_id')
_VOC = json.load(open(P_VOCAB)); V_MIC = int(_VOC['V']) + 1; PAD_ID = V_MIC - 1
_mic = _MIC.reindex(_sid)
_ids = _mic.micro_org_ids.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
_cnt = _mic.micro_org_cnt.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
MIC_L = max(1, int(_ids.map(len).max())); assert MIC_L <= 64
MIDS = np.full((len(ORDER), MIC_L), PAD_ID, 'int64'); MCNT = np.zeros((len(ORDER), MIC_L), 'float32')
for r, (a, b) in enumerate(zip(_ids, _cnt)):
    if a: MIDS[r, :len(a)] = a; MCNT[r, :len(a)] = b

XGF = np.load(P_OOF).astype('float32')
assert XGF.shape == (len(ORDER), C) and np.isfinite(XGF).all()
_l = np.log(np.clip(XGF, 1e-6, 1-1e-6) / (1 - np.clip(XGF, 1e-6, 1-1e-6)))
XGB_F = np.clip((_l - _l[TF_].mean(0, keepdims=True)) / (_l[TF_].std(0, keepdims=True) + 1e-6),
                -EHR_CLIP, EHR_CLIP).astype('float32')
FEAT = np.concatenate([EHR_COL, XGB_F], 1)      # `both` = 217 cot EHR + 40 cot XGB (53: tot nhat)
print(f'FEAT {FEAT.shape} = {EHR_COL.shape[1]} EHR + {C} XGB | tui vi khuan V={V_MIC-1} dai {MIC_L}')

_BASE = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_base.parquet'))
_BASE['index_time'] = pd.to_datetime(_BASE.index_time)
_B = (_BASE[_BASE.study_id.isin(set(ORDER.study_id.unique()))]
      .sort_values(['subject_id','index_time']).reset_index(drop=True))
_B['prev_study'] = _B.groupby('subject_id', sort=False).study_id.shift(1)
_PV = dict(zip(_B.study_id.to_numpy(), _B.prev_study.to_numpy()))
_rw = {}
for _i, _s in enumerate(_sid): _rw.setdefault(_s, []).append(_i)
PREV_ROW = np.full(len(ORDER), -1, 'int64')
for _i, _s in enumerate(_sid):
    _p2 = _PV.get(_s, np.nan)
    if not (isinstance(_p2, float) and np.isnan(_p2)): PREV_ROW[_i] = _rw[int(_p2)][0]
_hp = PREV_ROW >= 0
print(f'anh co phim truoc: {_hp.sum():,} / {len(ORDER):,} ({_hp.mean():.1%})  (44a do 67.2%)')
assert _hp.mean() > 0.5 and (PREV_ROW[_hp] != np.arange(len(ORDER))[_hp]).all()

## C2 · Ảnh + backbone + **cổng cắt đôi forward**

Rủi ro lớn nhất: cắt HF model sai chỗ ⇒ mọi số vô nghĩa **mà không báo lỗi**. `39` đã dính đúng chỗ
này — `Dinov2Layer` trả về **Tensor trần**, không phải tuple, nên `layer(h)[0]` lấy **mẫu đầu của
batch**. Hàm `_out()` dưới đây kiểm `isinstance` trước khi lấy `[0]`.

In [ ]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T

os.makedirs(LOCAL, exist_ok=True)
_n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
print(f'anh local: {_n:,} / {len(ORDER):,}')
if _n < len(ORDER) * 0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')
    assert os.path.isdir(SHARD_DIR), f'khong thay {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL, '_done.json')
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        nm, blob = pl; dst = os.path.join(LOCAL, os.path.basename(nm))
        if os.path.exists(dst): return
        Image.open(_io.BytesIO(blob)).convert('L').resize((SIZE, SIZE), Image.BICUBIC) \
             .save(dst, 'JPEG', quality=95)
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm([t for t in TARS if os.path.basename(t) not in done], desc='stage', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()   # doc BULK; lap tren mount FUSE se TREO
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE, 'w'))
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
assert _n >= len(ORDER) * 0.999, f'thieu {len(ORDER)-_n:,} anh'

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])
eval_tf = T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])

_ck = torch.load(BB_SRC, map_location='cpu', weights_only=False)
_sd0 = {k: v.clone() for k, v in AutoModel.from_pretrained('microsoft/rad-dino').state_dict().items()}
backbone = AutoModel.from_pretrained('microsoft/rad-dino')
backbone.load_state_dict(_ck['bb'], strict=True)
with torch.no_grad():
    _d = max((_sd0[k] - v).abs().max().item() for k, v in backbone.state_dict().items())
del _sd0
assert _d > 1e-6, f'backbone GIONG HET pretrain (lech {_d:.1e}) -> `ft` khong duoc nap'
backbone = backbone.to(dev).eval()
for p in backbone.parameters(): p.requires_grad_(False)
D_CXR = backbone.config.hidden_size
LAYERS = backbone.encoder.layer; NL = len(LAYERS)
assert max(ADAPT_AFTER) < NL, f'ADAPT_AFTER {ADAPT_AFTER} vuot {NL} block'
GRAD_FROM = min(ADAPT_AFTER)
print(f'BACKBONE `ft` 38a/{BB_RUN} | {NL} block | d={D_CXR} | lech pretrain {_d:.3f}')

def _out(r):
    '''Dinov2Layer tra ve Tensor TRAN o ban transformers nay. `r[0]` tren Tensor lay MAU DAU
       cua batch -> [B,N,D] thanh [N,D], broadcasting che lap loi nen KHONG bao gi. `39` da dinh.'''
    return r[0] if isinstance(r, tuple) else r
def run_blocks(h, lo, hi):
    for i in range(lo, hi): h = _out(LAYERS[i](h))
    return h

class DS(Dataset):
    '''Tra ve CA HAI anh: hien tai va PHIM TRUOC (+ `has`). `ehr_perm` cho TRAO NHAN QUA.'''
    def __init__(s, idx, tf, ehr_perm=None): s.idx, s.tf, s.perm = np.asarray(idx), tf, ehr_perm
    def __len__(s): return len(s.idx)
    def _img(s, i):
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        return s.tf(Image.merge('RGB', (im, im, im)))
    def __getitem__(s, k):
        i = int(s.idx[k]); pr = int(PREV_ROW[i]); x = s._img(i)
        return dict(image=x, image_prev=(s._img(pr) if pr >= 0 else torch.zeros_like(x)),
                    has=torch.tensor(1.0 if pr >= 0 else 0.0), pos=k,
                    y=torch.from_numpy(YM[i]), row=i,
                    erow=(i if s.perm is None else int(s.perm[k])))

def loader(idx, tf, shuffle, bs, perm=None, nw=NW, drop_last=None):
    return DataLoader(DS(idx, tf, perm), batch_size=bs, shuffle=shuffle, num_workers=nw,
                      pin_memory=True, drop_last=shuffle if drop_last is None else drop_last,
                      persistent_workers=nw > 0, prefetch_factor=4 if nw > 0 else None)

# --- cong 1: forward CAT DOI phai bang forward NGUYEN VEN ---
_b0 = next(iter(loader(I_VA[:4], eval_tf, False, 4, nw=0))); _x = _b0['image'].to(dev)
with torch.no_grad():
    _full = backbone(pixel_values=_x).last_hidden_state
    _split = backbone.layernorm(run_blocks(backbone.embeddings(_x), 0, NL))
_e1 = float((_full - _split).abs().max())
assert _e1 < 1e-3, (f'forward CAT DOI lech {_e1:.2e} -> da cat SAI cho trong kien truc HF. '
                    'DUNG chay tiep, moi so sau day vo nghia.')
N_TOK = _full.shape[1] - 1
assert N_TOK == (SIZE // backbone.config.patch_size) ** 2
print(f'cong 1: forward cat doi khop toi {_e1:.1e} | {N_TOK} patch token')
print(f'  block 0..{GRAD_FROM-1} no_grad (CHUNG) | block {GRAD_FROM}..{NL-1} co graph (RIENG arm adapter)')

## C3 · Kiến trúc — `53` + `FiLMAdapter`

In [ ]:
class GroupFC(nn.Module):
    def __init__(s, C, d):
        super().__init__()
        s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=2048, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a); return s.n2(h + s.ff(h)), w

class MicroBag(nn.Module):
    '''`const=True`: MOT chi so, khong phai MIC_L ban sao (tui that TB 1.7 vi khuan).'''
    def __init__(s, V, d, const=False):
        super().__init__()
        s.const = const; s.w = nn.Parameter(torch.randn(V, d) * d ** -0.5)
        s.mlp = nn.Sequential(nn.GELU(), nn.Linear(d, d))
    def forward(s, ids, w):
        if s.const:
            ids = torch.zeros(ids.shape[0], 1, dtype=torch.long, device=ids.device)
            w = torch.ones(ids.shape[0], 1, device=ids.device)
        return s.mlp(F.embedding_bag(ids, s.w, per_sample_weights=w, mode='sum'))

class ColTokBag(nn.Module):
    '''d_in token COT + 1 TONG HOP + 1 BAG. Chuoi nay CHUA k1 nhu truong hop rieng.'''
    def __init__(s, d_in, d=256, const=False, V=None):
        super().__init__()
        s.W = nn.Parameter(torch.randn(d_in, d) * d ** -0.5)
        s.b = nn.Parameter(torch.zeros(d_in, d))
        s.joint = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d); s.const = const
        if const: s.c = nn.Parameter(torch.randn(d_in + 1, d) * 0.02)
        s.mb = MicroBag(V, d, const) if V else None
        if s.mb is not None:
            s.bln = nn.LayerNorm(d); s.cln = nn.LayerNorm(d) if const else nn.Identity()
    def forward(s, e, mids=None, mcnt=None):
        if s.const: base = s.c.unsqueeze(0).expand(e.shape[0], -1, -1)
        else:
            col = s.b.unsqueeze(0) + e.unsqueeze(-1) * s.W.unsqueeze(0)
            base = s.ln(torch.cat([col, s.joint(e).unsqueeze(1)], 1))
        if s.mb is None: return base
        return torch.cat([s.cln(base), s.bln(s.mb(mids, mcnt)).unsqueeze(1)], 1)

class Tier1(nn.Module):
    '''Q = 1369 PATCH, K/V = token EHR. `pr` zero-init; `ln_a` chuan hoa DAU VAO cua phep chieu
       (ban va 38d — thieu no thi arm doi chung sup 0.2416 -> 0.0681).'''
    def __init__(s, d, nhead=8, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d); s.pr = nn.Linear(d, d)
        nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
    def forward(s, patch, etok):
        a, _ = s.at(s.n(patch), etok, etok, need_weights=False)
        return patch + s.pr(s.ln_a(a))

class Tier1Img(nn.Module):
    '''Phim truoc o muc PATCH, chay o khong gian D_CXR (TRUOC `pin`).
       KHONG tru thang `patch - prior`: hai phim khong duoc dang ky (tu the, do hit vao), o `i`
       co the la dinh phoi o phim nay va xuong don o phim kia. Cross-attention tu tim tuong ung.'''
    def __init__(s, d, nhead=8, drop=0.1, diff=True):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.nq = nn.LayerNorm(d); s.nk = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d)
        s.pr = nn.Linear(d, d); nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
        s.diff = diff
    def forward(s, patch, prior, has):
        a, _ = s.at(s.nq(patch), s.nk(prior), s.nk(prior), need_weights=False)
        z = s.pr(s.ln_a(patch - a if s.diff else a))
        return patch + z * has.view(-1, 1, 1)      # khong co phim truoc -> mask nhanh di han

class FiLMAdapter(nn.Module):
    '''Z' = Z * (1 + gamma(e)) + beta(e), e = mean(token EHR). Dieu bien TOAN CUC nhung dat TRONG
       backbone: `Z'` con di qua 4 block attention nua, ma attention la ham PHI TUYEN cua `Z`,
       nen cung phep toan lai la lop ham khac han so voi dat o dau ra (`06`: FiLM ~ concat,
       p = 0.308, DO O DAU RA). Bottleneck giu dang cua `11` de so voi G_mid = +0.0036.
       `const=True`: adapter nhan VECTOR HANG SO — Tier1 VAN nhan EHR that, nen `G_deep` co lap
       rieng phan dong gop cua adapter.'''
    def __init__(s, d, d_e, hid=256, const=False):
        super().__init__()
        s.const = const
        if const: s.c = nn.Parameter(torch.randn(d_e) * 0.02)
        s.mlp = nn.Sequential(nn.Linear(d_e, hid), nn.GELU(), nn.Linear(hid, 2 * d))
        nn.init.zeros_(s.mlp[-1].weight); nn.init.zeros_(s.mlp[-1].bias)   # zero-init -> Z' = Z
    def forward(s, z, etok):
        e = s.c.unsqueeze(0).expand(etok.shape[0], -1) if s.const else etok.mean(1)
        g, b = s.mlp(e).chunk(2, -1)
        return z * (1 + g.unsqueeze(1)) + b.unsqueeze(1)

class MLDecoder(nn.Module):
    '''Q = C query lop (KHONG hoc, dung theo bai goc). K/V = patch, co the da bi Tier1 dieu bien.'''
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1, d_ehr_in=0, t1=False):
        super().__init__()
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)
        s.pin = nn.Linear(d_in, d); s.img = XAttn(d, nhead, ff, drop)
        s.t1 = Tier1(d, nhead, drop) if t1 else None
        if t1: s.ein = nn.Linear(d_ehr_in, d)
        s.out = GroupFC(C, d)
    def forward(s, patch, etok=None, need_w=False):
        q = s.q.weight.unsqueeze(0).expand(patch.shape[0], -1, -1)
        kv = s.pin(patch)
        if s.t1 is not None: kv = s.t1(kv, s.ein(etok))
        h, w = s.img(q, kv, need_w); return s.out(h), w

class ASL(nn.Module):
    '''Verbatim nb11/12/13/15/35/37/38*/44*/45/52*/53. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()
print('kien truc san sang')

## C4 · 5 arm + cổng khoá trước

In [ ]:
ARMS = {
  'mldec':                dict(ehr=False, img=False, const=False, film=None),
  't1_prior_both':        dict(ehr=True,  img=True,  const=False, film=None),
  't1_prior_both_ehrz':   dict(ehr=True,  img=True,  const=True,  film=None),
  't1_prior_both_film':   dict(ehr=True,  img=True,  const=False, film='real'),
  't1_prior_both_film_u': dict(ehr=True,  img=True,  const=False, film='const'),
}
ADP_ARMS = [t for t, c in ARMS.items() if c['film']]

def build_arms(seed=SEED):
    M, E, I, A = nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict()
    for tag, cfg in ARMS.items():
        torch.manual_seed(seed)     # gieo lai TRUOC TUNG arm -> khoi tao giong het nhau
        if cfg['ehr']:
            E[tag] = ColTokBag(FEAT.shape[1], D_ETOK, cfg['const'], V=V_MIC)
        M[tag] = MLDecoder(C, D_CXR, D_MLD, d_ehr_in=D_ETOK, t1=cfg['ehr'])
        if cfg['img']: I[tag] = Tier1Img(D_CXR)
        if cfg['film']:
            A[tag] = nn.ModuleList([FiLMAdapter(D_CXR, D_ETOK, FILM_HID,
                                                const=(cfg['film'] == 'const'))
                                    for _ in ADAPT_AFTER])
    return M.to(dev), E.to(dev), I.to(dev), A.to(dev)

MOD, ETK, IMG, ADP = build_arms()
NP_ = {t: sum(p.numel() for m in ([MOD[t]] + [x[t] for x in (ETK, IMG, ADP) if t in x])
              for p in m.parameters() if p.requires_grad) for t in ARMS}
print(f'{"arm":<24} {"tham so":>12} {"adapter":>10}')
for t in ARMS:
    na = sum(p.numel() for p in ADP[t].parameters()) if t in ADP else 0
    print(f'  {t:<22} {NP_[t]:>12,} {na:>10,}')
print(f'  (`11` dung bottleneck 234,368 tham so cho adapter)')

_e = torch.randn(4, FEAT.shape[1], device=dev)
_mi = torch.zeros(4, MIC_L, dtype=torch.long, device=dev); _mc = torch.zeros(4, MIC_L, device=dev)
_z = torch.randn(4, N_TOK + 1, D_CXR, device=dev); _p = torch.randn(4, N_TOK, D_CXR, device=dev)
_pp = torch.randn(4, N_TOK, D_CXR, device=dev)
_h1 = torch.ones(4, device=dev); _h0 = torch.zeros(4, device=dev)
tok = lambda t: ETK[t](_e, _mi, _mc)

assert not any(p.requires_grad for p in backbone.parameters()), 'backbone chua dong cung'
assert not MOD['mldec'].q.weight.requires_grad, 'query_embed dang hoc'
with torch.no_grad():
    m = MOD['t1_prior_both'].eval(); l1, _ = m(_p, tok('t1_prior_both'))
    _k = m.t1; m.t1 = None; l2, _ = m(_p, tok('t1_prior_both')); m.t1 = _k
    assert (l1 - l2).abs().max() < 1e-5, 'Tier1 KHONG zero-init'
    for t in IMG: assert (IMG[t](_p, _pp, _h1) - _p).abs().max() < 1e-6, f'{t}: Tier1Img khong zero-init'
    for t in ADP:
        for a in ADP[t]:
            assert (a(_z, tok(t)) - _z).abs().max() < 1e-6, f'{t}: adapter KHONG zero-init'
print('\ncong 2: Tier1 · Tier1Img · adapter deu zero-init OK')
with torch.no_grad():
    _ti = IMG['t1_prior_both']; _ti.pr.weight.normal_(0, 0.02)
    _r = float((_ti(_p, _pp*100.0, _h1) - _p).abs().max() / ((_ti(_p, _pp, _h1) - _p).abs().max()+1e-9))
    assert (_ti(_p, _pp, _h0) - _p).abs().max() < 1e-6, 'has=0 KHONG cat duoc nhanh phim truoc'
    nn.init.zeros_(_ti.pr.weight)
assert _r < 3.0, f'phong dai K/V 100x -> bien do ra tang {_r:.1f}x, `ln_a` khong chan duoc'
print(f'cong 3: `ln_a` chan bien do OK ({_r:.2f}x) | has=0 cat sach nhanh OK')
for a, b in (('t1_prior_both','t1_prior_both_ehrz'), ('t1_prior_both_film','t1_prior_both_film_u')):
    assert NP_[b] >= NP_[a], f'{b} IT tham so hon {a} -> doi chung khong con bao thu'
    print(f'cong 4: {b} {NP_[b]:,} >= {a} {NP_[a]:,} (+{NP_[b]-NP_[a]:,}) -> `{a}-{b}>0` BAO THU')
MOD, ETK, IMG, ADP = build_arms()

## C5 · Chấm điểm — verbatim `53`

In [ ]:
from sklearn.metrics import average_precision_score
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}
_rs6 = np.random.default_rng(6); _yy = (_rs6.random((3000, C)) < 0.2).astype('float32')
_ss6 = _rs6.random((3000, C)).astype('float32')
_dd6 = float(np.nanmax(np.abs(ap_cols(_yy, _ss6) -
        np.array([average_precision_score(_yy[:, j], _ss6[:, j]) for j in range(C)]))))
assert _dd6 < 1e-6, f'ap_cols lech sklearn {_dd6:.2e}'
print(f'cong 5: ap_cols khop sklearn toi {_dd6:.2e}')

## C6 · Huấn luyện

Adapter áp cho **CẢ** phim hiện tại **và** phim trước — nếu chỉ điều biến phim hiện tại thì
`patch − a` của `Tier1Img` chứa chính phần điều biến, làm nhiễm nhánh phim trước.

Ghi **mỗi epoch**: `pr_norm`, `clip_pct`, `ratio ‖ΔZ‖/‖Z‖`. `53` ghi đè checkpoint mỗi epoch nên
chỉ còn trạng thái cuối — vì thế các cú sụp ở đó không truy được nguyên nhân.

In [ ]:
crit = ASL(gn=4, gp=0).to(dev)
MIDS_T, MCNT_T = torch.from_numpy(MIDS), torch.from_numpy(MCNT)
FEAT_T = torch.from_numpy(FEAT)

@torch.no_grad()
def shared_lo(img):
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        return run_blocks(backbone.embeddings(img), 0, GRAD_FROM).float()

def upper(h, t=None, et=None):
    '''block GRAD_FROM..NL-1. Neu `t` co adapter thi chen adapter va GIU graph.'''
    ai = 0
    for i in range(GRAD_FROM, NL):
        if t is not None and i in ADAPT_AFTER: h = ADP[t][ai](h, et); ai += 1
        h = _out(LAYERS[i](h))
    return backbone.layernorm(h)[:, 1:]

def arm_forward(t, hl, hlp, erow, has):
    cfg = ARMS[t]
    et = ETK[t](FEAT_T[erow].to(dev), MIDS_T[erow].to(dev), MCNT_T[erow].to(dev)) \
         if cfg['ehr'] else None
    if cfg['film']:
        p  = upper(hl,  t, et)          # adapter cho CA HAI anh — xem ghi chu o tren
        pp = upper(hlp, t, et)
    else:
        with torch.no_grad(): p, pp = upper(hl), upper(hlp)
    if cfg['img']: p = IMG[t](p, pp, has)
    return MOD[t](p, et)[0]

@torch.no_grad()
def predict(idx, perm=None, bs=32):
    MOD.eval(); ETK.eval(); IMG.eval(); ADP.eval()
    P = {t: np.zeros((len(idx), C), 'float32') for t in ARMS}
    for b in tqdm(loader(idx, eval_tf, False, bs, perm, nw=0, drop_last=False),
                  desc='eval', leave=False):
        hl = shared_lo(b['image'].to(dev, non_blocking=True))
        hlp = shared_lo(b['image_prev'].to(dev, non_blocking=True)); hs = b['has'].to(dev)
        for t in ARMS:
            P[t][b['pos'].numpy()] = torch.sigmoid(
                arm_forward(t, hl, hlp, b['erow'], hs).float()).cpu().numpy()
    assert not any(np.isnan(v).any() for v in P.values()), 'predict cho ra NaN'
    for t in ARMS: assert np.abs(P[t]).sum(1).min() > 0, f'{t}: hang khong duoc ghi'
    return P

P_LOG = os.path.join(RES56, 'epoch_log.json')
LOG = json.load(open(P_LOG)) if os.path.exists(P_LOG) else {}
DL = loader(I_TF, train_tf, True, BATCH)

for seed in SEEDS:
    sk = f's{seed}'
    if LOG.get(sk, {}).get('done'): print(f'{sk}: da xong, bo qua'); continue
    MOD, ETK, IMG, ADP = build_arms(seed)
    PARAM_OF = {t: [p for m in ([MOD[t]] + [x[t] for x in (ETK, IMG, ADP) if t in x])
                    for p in m.parameters() if p.requires_grad] for t in ARMS}
    N_CLIP = {t: 0 for t in ARMS}; N_SKIP = {t: 0 for t in ARMS}; N_STEP = 0
    rest = [p for m in (MOD, ETK, IMG, ADP) for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW([{'params': [p for p in rest if p.ndim > 1], 'weight_decay': WD},
                             {'params': [p for p in rest if p.ndim <= 1], 'weight_decay': 0.0}], lr=LR)
    steps = math.ceil(len(I_TF) / (BATCH*ACCUM)) * EPOCHS; warm = int(WARMUP*steps)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/max(warm,1) if s < warm else
                                            0.5*(1+math.cos(math.pi*(s-warm)/max(steps-warm,1))))
    LOG.setdefault(sk, {})
    P_RES = os.path.join(CKPT, f'{sk}_resume.pt'); ep0 = 0
    if os.path.exists(P_RES):
        _r = torch.load(P_RES, map_location=dev, weights_only=False)
        _old = _r.get('arms')
        assert _old is None or list(_old) == list(ARMS), (
            f'checkpoint co {len(_old)} arm {list(_old)}, notebook co {len(ARMS)} arm.\n'
            '  -> DOI RUN_TAG o C0 (vd "r2") roi chay lai tu C0. KHONG xoa gi.')
        MOD.load_state_dict(_r['mod']); ETK.load_state_dict(_r['etk'])
        IMG.load_state_dict(_r['img']); ADP.load_state_dict(_r['adp'])
        opt.load_state_dict(_r['opt']); sch.load_state_dict(_r['sch'])
        N_CLIP, N_SKIP, N_STEP, ep0 = _r['n_clip'], _r['n_skip'], _r['n_step'], _r['ep'] + 1
        print(f'{sk}: resume tu epoch {ep0+1}/{EPOCHS}')

    for ep in range(ep0, EPOCHS):
        MOD.train(); ETK.train(); IMG.train(); ADP.train()
        t0, tot, nb = time.time(), {t: 0.0 for t in ARMS}, 0
        opt.zero_grad(set_to_none=True)
        for j, b in enumerate(tqdm(DL, desc=f'{sk} ep{ep+1}/{EPOCHS}', leave=False)):
            hl = shared_lo(b['image'].to(dev, non_blocking=True))
            hlp = shared_lo(b['image_prev'].to(dev, non_blocking=True))
            y = b['y'].to(dev); hs = b['has'].to(dev)
            loss, bad = 0.0, []
            for t in ARMS:
                lg = arm_forward(t, hl, hlp, b['erow'], hs)
                if not torch.isfinite(lg).all(): bad.append(t); N_SKIP[t] += 1; continue
                l_ = crit(lg.float(), y); loss = loss + l_; tot[t] += float(l_.detach())
            if bad and (sum(N_SKIP.values()) <= 3 or j % 400 == 0):
                print(f'   [{sk} ep{ep+1} buoc {j}] logit khong huu han: {bad}')
            if not torch.is_tensor(loss): opt.zero_grad(set_to_none=True); continue
            (loss / ACCUM).backward(); nb += 1
            if (j + 1) % ACCUM == 0:
                # Kiem gradient TRUOC khi buoc: total_norm = nan cho clip_coef = nan va BOI BAN
                # toan bo tham so -> mot buoc hong lam chet vinh vien.
                for t, ps in PARAM_OF.items():
                    gn = torch.nn.utils.clip_grad_norm_(ps, CLIP, error_if_nonfinite=False)
                    if not torch.isfinite(gn):
                        for p in ps:
                            if p.grad is not None: p.grad.zero_()
                        N_SKIP[t] += 1
                    elif float(gn) > CLIP: N_CLIP[t] += 1
                N_STEP += 1; opt.step(); sch.step(); opt.zero_grad(set_to_none=True)

        PV = predict(I_VA); PS = predict(I_SL); ensure_res()
        for t, p in PV.items(): np.save(os.path.join(RES56, f'p_{t}_{sk}_ep{ep+1}.npy'), p)
        for t, p in PS.items(): np.save(os.path.join(RES56, f'psl_{t}_{sk}_ep{ep+1}.npy'), p)
        if DO_TEST and ep >= EPOCHS - N_TEST_EP:
            PT = predict(I_TE); ensure_res()
            for t, p in PT.items(): np.save(os.path.join(RES56, f'pte_{t}_{sk}_ep{ep+1}.npy'), p)
            with open(P_TOUCH, 'a') as _f:
                _f.write(json.dumps(dict(nb='56', run=RUN_TAG, seed=seed, ep=ep+1, n=len(I_TE),
                                         arms=list(ARMS), muc_dich='theo doi',
                                         t=datetime.datetime.now().isoformat(timespec='seconds')))+'\n')
            print(f'   [TEST] infer {len(I_TE):,} anh -> pte_*_ep{ep+1}.npy')
        # --- do CO CHE moi epoch: `53` ghi de checkpoint nen cac cu sup khong truy duoc ---
        with torch.no_grad():
            _bd = next(iter(loader(I_VA[:32], eval_tf, False, 32, nw=0)))
            _hl = shared_lo(_bd['image'].to(dev)); _rt = {}
            for t in ADP_ARMS:
                _et = ETK[t](FEAT_T[_bd['erow']].to(dev), MIDS_T[_bd['erow']].to(dev),
                             MCNT_T[_bd['erow']].to(dev))
                _z1 = ADP[t][0](_hl, _et)
                _rt[t] = float(((_z1-_hl).norm(dim=-1) / (_hl.norm(dim=-1)+1e-9)).mean())
        _e2 = {}
        for t, p in PV.items():
            ap = ap_cols(YM[VA], p)
            _e2[t] = dict(mAP_anh=float(np.nanmean(ap)), **tiers(ap),
                          sel=float(np.nanmean(ap_cols(YM[SL], PS[t]))),
                          pr_norm=float(IMG[t].pr.weight.norm()) if t in IMG else 0.0,
                          clip_pct=N_CLIP[t]/max(N_STEP, 1), ratio=_rt.get(t, 0.0))
        LOG[sk][str(ep)] = _e2; json.dump(LOG, open(P_LOG, 'w'))
        torch.save(dict(mod=MOD.state_dict(), etk=ETK.state_dict(), img=IMG.state_dict(),
                        adp=ADP.state_dict(), opt=opt.state_dict(), sch=sch.state_dict(),
                        n_clip=N_CLIP, n_skip=N_SKIP, n_step=N_STEP, ep=ep, seed=seed,
                        arms=list(ARMS)), P_RES)
        g = lambda a, b: _e2[a]['mAP_anh'] - _e2[b]['mAP_anh']
        sh = lambda t: t.replace('t1_prior_both', 'tpb')
        print(f'{sk} ep{ep+1:>2} | {nb*BATCH/max(time.time()-t0,1e-9):.0f} img/s | '
              f'{(time.time()-t0)/60:.1f} ph | loss ' +
              ' '.join(f'{sh(t)}:{tot[t]/max(nb,1):.3f}' for t in ARMS))
        print('   VAL      ' + ' '.join(f'{sh(t):>10}:{_e2[t]["mAP_anh"]:.4f}' for t in ARMS))
        print('   LAT CHON ' + ' '.join(f'{sh(t):>10}:{_e2[t]["sel"]:.4f}' for t in ARMS))
        print('   TAIL     ' + ' '.join(f'{sh(t):>10}:{_e2[t]["tail"]:.4f}' for t in ARMS))
        print('   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`')
        print(f'   G_ehr     = tpb - tpb_ehrz    {g("t1_prior_both","t1_prior_both_ehrz"):+.4f}'
              '   (53 do +0.0060)')
        print(f'   G_deep    = tpb_film - tpb_film_u  '
              f'{g("t1_prior_both_film","t1_prior_both_film_u"):+.4f}   (39 do G_film +0.0020)')
        print(f'   CONG THEM = tpb_film - tpb        '
              f'{g("t1_prior_both_film","t1_prior_both"):+.4f}   <- CAU HOI CHINH')
        print('   pr_norm ' + ' '.join(f'{sh(t)}:{_e2[t]["pr_norm"]:.2f}' for t in ARMS if t in IMG)
              + ' | ratio ' + ' '.join(f'{sh(t)}:{_rt[t]:.4f}' for t in ADP_ARMS)
              + f' | clip {max(v["clip_pct"] for v in _e2.values()):.1%}')
        if max(N_SKIP.values()) > 0:
            print('   BUOC BI BO: ' + ' '.join(f'{t}:{v}' for t, v in N_SKIP.items() if v > 0))
    ensure_res()
    torch.save(dict(mod=MOD.state_dict(), etk=ETK.state_dict(), img=IMG.state_dict(),
                    adp=ADP.state_dict(), seed=seed, arms=list(ARMS)),
               os.path.join(CKPT, f'{sk}.pt'))
    LOG[sk]['done'] = True; json.dump(LOG, open(P_LOG, 'w'))
    print(f'{sk}: xong')
print('tat ca seed xong')

## C7 · Kết quả + cổng + tráo nhân quả + nhóm nhãn + TEST

In [ ]:
LOG = json.load(open(P_LOG)); EPk = str(EPOCHS - 1)
R = pd.DataFrame([dict(seed=s, arm=t, **v) for s in SEEDS if EPk in LOG.get(f's{s}', {})
                  for t, v in LOG[f's{s}'][EPk].items()])
assert len(R), 'chua co ket qua nao'
SD = [s for s in SEEDS if EPk in LOG.get(f's{s}', {})]
piv = R.pivot_table(index='arm', values=['mAP_anh','head','medium','tail','sel','pr_norm','ratio'],
                    aggfunc='mean').reindex(list(ARMS))
piv['bien_do'] = R.groupby('arm').mAP_anh.agg(lambda s: s.max()-s.min()).reindex(list(ARMS))
piv['tham_so'] = [NP_[t] for t in ARMS]
print(f'=== TUYET DOI (TB qua {len(SD)} seed) ===')
print(piv.to_string(float_format=lambda v: f'{v:.4f}'))

g = lambda a, s: float(R[(R.arm == a) & (R.seed == s)].mAP_anh.iloc[0])
gt = lambda a, s: float(R[(R.arm == a) & (R.seed == s)]['tail'].iloc[0])
def d(a, b, ngh):
    v = [g(a, s) - g(b, s) for s in SD]; vt = [gt(a, s) - gt(b, s) for s in SD]
    bd = (max(v) - min(v)) if len(v) > 1 else float('nan')
    ok = len(v) > 1 and (all(x > 0 for x in v) or all(x < 0 for x in v))
    print(f'  {a:<22} - {b:<22} {np.mean(v):+.4f} (bien do {bd:.4f}, '
          f'{"DAT" if ok and abs(np.mean(v)) > bd else "---"}) | tail {np.mean(vt):+.4f}   {ngh}')
    return float(np.mean(v)), float(np.mean(vt))

print('\n=== NOI DUNG THUAN (arm - doi chung KHOP THAM SO) ===')
G_EHR, _  = d('t1_prior_both', 't1_prior_both_ehrz', 'nhanh EHR (53 do +0.0060)')
G_DEEP, T_DEEP = d('t1_prior_both_film', 't1_prior_both_film_u', 'ADAPTER (39 do G_film +0.0020)')
print('\n=== CAU HOI CHINH ===')
ADD, T_ADD = d('t1_prior_both_film', 't1_prior_both', 'deep CONG THEM gi TREN kien truc chinh')
print('\n' + '=' * 72)
print('CONG KHOA TRUOC, dat tren G_deep VA tail:')
print('  G_deep < +0.002        -> DONG huong deep conditioning cho 40 nhan CXR-LT')
print('    +0.002 .. +0.004     -> vung xam, can seed thu ba')
print('  G_deep > +0.004        -> mo tiep')
print('  VA tail(CONG THEM) < -0.003  -> DONG bat ke macro (39 do tail -0.0050)')
_v = 'DONG' if G_DEEP < 0.002 else ('vung xam' if G_DEEP < 0.004 else 'MO TIEP')
print(f'\n  G_deep {G_DEEP:+.4f} -> {_v}   |   tail(CONG THEM) {T_ADD:+.4f} -> '
      f'{"DONG (hai tail)" if T_ADD < -0.003 else "khong hai tail"}')
print('=' * 72)
print(f'\nmoc: 39 G_film +0.0020, tail -0.0050 (KHONG co Tier1Img) | 53 t1_prior_both 0.2560')
ensure_res(); R.to_csv(os.path.join(RES56, 'main_table.csv'), index=False)

ck = torch.load(os.path.join(CKPT, f's{SD[0]}.pt'), map_location=dev, weights_only=False)
MOD.load_state_dict(ck['mod']); ETK.load_state_dict(ck['etk'])
IMG.load_state_dict(ck['img']); ADP.load_state_dict(ck['adp'])
perm = np.random.default_rng(0).permutation(len(I_VA))
P0 = predict(I_VA); P1 = predict(I_VA, perm=I_VA[perm])
lgt = lambda p: np.log(p.clip(1e-6, 1-1e-6) / (1 - p.clip(1e-6, 1-1e-6)))
print(f'\n=== TRAO EHR SANG BENH NHAN KHAC (s{SD[0]}) — NHAN giu nguyen ===')
print('  Doc HAI cot: corr mot minh QUA CHAT (44b tung co corr = 1.000000 ma mat 0.0056 mAP).')
for t in ARMS:
    if not ARMS[t]['ehr']: continue
    r = float(np.corrcoef(lgt(P0[t]).ravel(), lgt(P1[t]).ravel())[0, 1])
    dd = float(np.nanmean(ap_cols(YM[VA], P1[t])) - np.nanmean(ap_cols(YM[VA], P0[t])))
    print(f'  {t:<24} corr {r:.6f} | mAP khi trao {dd:+.4f}   '
          + ('EHR co tac dong that' if dd <= -0.002 else
             'EHR KHONG duoc dung' if r > 0.9999 and dd > -0.0005 else 'vung xam'))

INF = [l for l in LAB40 if re.search(r'pneumon|tubercul|infect|consolidat|abscess|empyema', l, re.I)]
I_INF = np.array([LAB40.index(l) for l in INF]); sk = f's{SD[0]}'; _r = []
for t in ARMS:
    f = os.path.join(RES56, f'p_{t}_{sk}_ep{EPOCHS}.npy')
    if not os.path.exists(f): continue
    a = ap_cols(YM[VA], np.load(f))
    _r.append(dict(arm=t, macro=float(np.nanmean(a)), nhiem_trung=float(np.nanmean(a[I_INF])),
                   tail=float(np.nanmean(a[TIER['tail']]))))
T = pd.DataFrame(_r).set_index('arm')
for c in ('macro','nhiem_trung','tail'): T[f'd_{c}'] = T[c] - T.loc['t1_prior_both', c]
print(f'\nnhom nhiem trung ({len(INF)}) — hieu so voi `t1_prior_both`')
print(T.to_string(float_format=lambda v: f'{v:+.4f}'))
print('  39 do: adapter tap trung o nhiem trung (+0.0098) nhung HAI tail (-0.0050).')

if DO_TEST:
    _nl = int((YM[TE].sum(0) > 0).sum()); _rt2 = float(np.nanmean(YM[TE].mean(0)/(YM[VA].mean(0)+1e-9)))
    print(f'\n=== TEST (CHI theo doi) === {_nl}/{C} nhan co ca duong | prev(test)/prev(val) {_rt2:.3f}')
    if _nl >= 35 and 0.5 <= _rt2 <= 2.0:
        rows = []
        for s in SD:
            for t in ARMS:
                fs = [os.path.join(RES56, f'pte_{t}_s{s}_ep{e}.npy')
                      for e in range(EPOCHS-N_TEST_EP+1, EPOCHS+1)]
                fs = [f for f in fs if os.path.exists(f)]
                if not fs: continue
                a = ap_cols(YM[TE], np.mean([np.load(f) for f in fs], 0))   # luat last-3 cua 48
                rows.append(dict(seed=s, arm=t, test_last3=float(np.nanmean(a)),
                                 test_tail=float(np.nanmean(a[TIER['tail']]))))
        if rows:
            TT = pd.DataFrame(rows).pivot_table(index='arm', values=['test_last3','test_tail'],
                                                aggfunc='mean')
            TT['val'] = [piv.loc[a, 'mAP_anh'] for a in TT.index]
            print(TT.to_string(float_format=lambda v: f'{v:.4f}'))
            ensure_res(); TT.to_csv(os.path.join(RES56, 'test_table.csv'))
    else: print('  !! nhan test co ve BI GIU LAI hoac lech -> KHONG doc so test')
    print(f'  !! model chi thay {int(TF_.sum()):,}/{int((TR|VA).sum()):,} anh co nhan (THIEU 26%).')
    print('     So test la CAN DUOI, KHONG phai con so cuoi cho bai (do thuoc NHANH B).')

## C8 · Ghi `REPORT_STAGE4_NB56.md`

In [ ]:
L = []; w = L.append
w('# REPORT · `nb56` — deep conditioning **trên kiến trúc chính** `t1_prior_both`'); w('')
w(f'`{RUN_TAG}` · backbone `ft` = `38a/{BB_RUN}` **đông cứng** · adapter sau block {ADAPT_AFTER} · '
  f'`asl40` · `E={EPOCHS}` · **epoch cuối** · seeds {list(SD)} · val {int(VA.sum()):,} ảnh.'); w('')
w('| arm | tham số | mAP ảnh | biên độ | tail | nhiễm trùng |'); w('|---|---|---|---|---|---|')
for t in ARMS:
    if t not in piv.index: continue
    r = piv.loc[t]
    w(f'| `{t}` | {NP_[t]:,} | {r["mAP_anh"]:.4f} | {r["bien_do"]:.4f} | {r["tail"]:.4f} | '
      + (f'{T.loc[t, "nhiem_trung"]:.4f}' if t in T.index else '—') + ' |')
w(''); w('```')
w(f'G_ehr     = tpb - tpb_ehrz         {G_EHR:+.4f}   (53 do +0.0060)')
w(f'G_deep    = tpb_film - tpb_film_u  {G_DEEP:+.4f}   (39 do +0.0020, KHONG co Tier1Img)')
w(f'CONG THEM = tpb_film - tpb         {ADD:+.4f}   tail {T_ADD:+.4f}')
w('```'); w('')
w('**Vì sao chạy lại thứ `39` đã đo.** `39` đo trên `mldec + adapter`, **không có `Tier1Img`**. Và '
  '`52b` (không `Tier1Img`) cho `nội dung(col) = −0.0015` trong khi `53` (có) cho `+0.0060` — cùng '
  'bộ mã hoá, hai dấu ngược nhau. Nên `+0.0020` của `39` rất có thể là **đọc hụt**.'); w('')
w('## Những gì kết quả này KHÔNG trả lời'); w('')
w(f'- **Một vị trí duy nhất** (sau block {ADAPT_AFTER}) và **một hình dạng** (FiLM). `39` đã đo '
  '`G_xattn − G_film = −0.0003` ⇒ hình dạng không quan trọng, nên chỉ chạy FiLM ở đây.')
w('- **`p_uni` chưa chạm** — tỉ lệ mẫu uni-modal là thay đổi **lấy mẫu** của cả vòng lặp, không làm '
  'được thành một arm. MedFuse đo `+0.015` từ đúng đòn bẩy này.')
w('- **Backbone vẫn đông cứng dưới block 8.** Adapter chỉ can thiệp từ block 8 trở lên.')
w(f'- **Số test là CẬN DƯỚI** (model thiếu 26% dữ liệu có nhãn). Test **không** tham gia lựa chọn '
  'nào; mọi lần chạm ghi ở `_test_touches.jsonl`.')
w('- **Đích.** Nhãn CXR-LT do NLP trích từ **chính report của phim đó** ⇒ ảnh gần như đủ theo định '
  'nghĩa (`34`: nhãn tự động gọi thừa `Pneumonia` **5×**).')
TXT = '\n'.join(L); ensure_res()
open(os.path.join(RES56, 'REPORT_STAGE4_NB56.md'), 'w', encoding='utf-8').write(TXT)
print(TXT)